# ThermoRG Phase B — Quick Validation (10 Epochs)
## Goal
Verify that J_topo ranking correlates with early training loss.

**Candidates:** T18, T04, T11, T10
**Training:** 10 epochs each on CIFAR-10
**Metric:** Validation loss after 10 epochs

If J_topo predicts E_floor, the ranking by J_topo should match the ranking by validation loss.

In [ ]:
import os, sys, time, json, math
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

from datetime import datetime

import warnings
warnings.filterwarnings('ignore')

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {DEVICE}")

In [ ]:
# ============================================================
# ThermoNet Model Builders (from phase_a_dscaling.py)
# ============================================================

def scale_channels(chs, mult):
    return [chs[0]] + [max(1, int(c * mult)) for c in chs[1:]]

class ConvBlock(nn.Module):
    def __init__(self, ic, oc, act='gelu', norm=True):
        super().__init__()
        self.conv = nn.Conv2d(ic, oc, 3, padding=1, bias=not norm)
        self.norm = nn.LayerNorm([oc, 32, 32]) if norm else nn.Identity()
        if act == 'gelu':
            self.act = nn.GELU()
        elif act == 'tga':
            self.act = nn.Tanh()
        else:
            self.act = nn.ReLU(inplace=True)
    
    def forward(self, x):
        return self.act(self.norm(self.conv(x)))

class SkipConnection(nn.Module):
    def __init__(self, ic, oc, s=1):
        super().__init__()
        if ic == oc and s == 1:
            self.skip = nn.Identity()
        else:
            self.skip = nn.Sequential(
                nn.Conv2d(ic, oc, 1, s, bias=False),
                nn.BatchNorm2d(oc)
            )
    def forward(self, x, residual):
        return x + self.skip(residual)

def build_TN3(wm=1.0, num_classes=10, use_skip=True):
    ch = scale_channels([3, 64, 64, 128, 128], wm)
    blocks = nn.ModuleList()
    for i in range(len(ch) - 1):
        blocks.append(ConvBlock(ch[i], ch[i+1], 'gelu', True))
    pool = nn.AdaptiveAvgPool2d((1, 1))
    fc = nn.Linear(ch[-1], num_classes)
    return nn.Sequential(*[*blocks, pool, nn.Flatten(), fc])

def build_TN5(wm=1.0, num_classes=10, use_skip=True):
    ch = scale_channels([3, 64, 128, 256, 128, 64], wm)
    blocks = nn.ModuleList()
    for i in range(len(ch) - 1):
        blocks.append(ConvBlock(ch[i], ch[i+1], 'gelu', True))
    pool = nn.AdaptiveAvgPool2d((1, 1))
    fc = nn.Linear(ch[-1], num_classes)
    return nn.Sequential(*[*blocks, pool, nn.Flatten(), fc])

def build_TN7(wm=1.0, num_classes=10, use_skip=True):
    ch = scale_channels([3, 64, 64, 128, 128, 256, 128, 64], wm)
    blocks = nn.ModuleList()
    for i in range(len(ch) - 1):
        blocks.append(ConvBlock(ch[i], ch[i+1], 'tga', True))
    pool = nn.AdaptiveAvgPool2d((1, 1))
    fc = nn.Linear(ch[-1], num_classes)
    return nn.Sequential(*[*blocks, pool, nn.Flatten(), fc])

def build_TN9(wm=1.0, num_classes=10, use_skip=False):
    ch = scale_channels([3] + [64]*8, wm)
    blocks = nn.ModuleList()
    for i in range(len(ch) - 1):
        blocks.append(ConvBlock(ch[i], ch[i+1], 'gelu', True))
    pool = nn.AdaptiveAvgPool2d((1, 1))
    fc = nn.Linear(ch[-1], num_classes)
    return nn.Sequential(*[*blocks, pool, nn.Flatten(), fc])

def build_TN_arbitrary_depth(depth, wm=1.0, num_classes=10, use_skip=True):
    if depth == 3:
        return build_TN3(wm, num_classes, use_skip)
    elif depth == 5:
        return build_TN5(wm, num_classes, use_skip)
    elif depth == 7:
        return build_TN7(wm, num_classes, use_skip)
    elif depth == 9:
        return build_TN9(wm, num_classes, use_skip)
    else:
        # For depth > 9
        base_pattern = [64, 128, 256, 512]
        ch = [3]
        reps = (depth + 3) // 4
        for _ in range(reps):
            ch.extend(base_pattern)
        ch = ch[:depth+1]
        while len(ch) < depth + 1:
            ch.append(64)
        ch = scale_channels(ch, wm)
        blocks = nn.ModuleList()
        for i in range(len(ch) - 1):
            blocks.append(ConvBlock(ch[i], ch[i+1], 'gelu', True))
        pool = nn.AdaptiveAvgPool2d((1, 1))
        fc = nn.Linear(ch[-1], num_classes)
        return nn.Sequential(*[*blocks, pool, nn.Flatten(), fc])

In [ ]:
# ============================================================
# J_topo Computation
# ============================================================

def compute_D_eff_power_iteration(W, n_iter=20):
    W_flat = W.reshape(W.shape[0], -1)
    if min(W_flat.shape) == 0:
        return 1.0
    v = torch.randn(W_flat.shape[1], device=W_flat.device)
    v = v / (v.norm() + 1e-10)
    for _ in range(n_iter):
        Wv = torch.matmul(W_flat.T, torch.matmul(W_flat, v))
        v_new = Wv / (Wv.norm() + 1e-10)
        if torch.abs(v - v_new).sum() < 1e-8:
            break
        v = v_new
    lambda_max_sq = torch.matmul(W_flat.T, torch.matmul(W_flat, v)).norm()**2 / (v.norm()**2 + 1e-10)
    fro_sq = (W_flat ** 2).sum()
    return float((fro_sq / (lambda_max_sq + 1e-10)).clamp(min=1.0))

def compute_J_topo(model):
    import re
    skip_exclude = ['layernorm', 'layer_norm', 'norm', 'batchnorm', 'bn', 
                    'pool', 'flatten', 'fc', 'linear']
    eta_list = []
    prev_D_eff = None
    
    for name, module in model.named_modules():
        if any(re.search(p, name.lower()) for p in skip_exclude):
            eta_list.append(1.0)
            continue
        if not hasattr(module, 'weight') or module.weight is None:
            continue
        W = module.weight.data
        if W.numel() == 0:
            continue
        W_flat = W.reshape(W.shape[0], -1)
        if min(W_flat.shape) == 0:
            continue
        D_eff = compute_D_eff_power_iteration(W, n_iter=20)
        if prev_D_eff is not None:
            eta = D_eff / max(prev_D_eff, 1.0)
            eta_list.append(float(eta))
        prev_D_eff = D_eff
    
    if not eta_list:
        return 1.0
    log_etas = [abs(math.log(max(eta, 1e-10))) for eta in eta_list]
    return math.exp(-np.mean(log_etas))

In [ ]:
# ============================================================
# Data Loading (with train/val split)
# ============================================================

transform_train = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize([0.4914, 0.4822, 0.4465], [0.2023, 0.1994, 0.2010])
])
transform_val = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize([0.4914, 0.4822, 0.4465], [0.2023, 0.1994, 0.2010])
])

# Full training set
full_train = datasets.CIFAR10(root='./data', train=True, download=True, transform=transform_train)

# Split: 90% train, 10% validation
n_total = len(full_train)
n_val = int(0.1 * n_total)
n_train = n_total - n_val
train_subset, val_subset = torch.utils.data.random_split(
    full_train, [n_train, n_val], 
    generator=torch.Generator().manual_seed(42)
)

# Re-apply normalization transform to val subset (random_split loses transform)
val_dataset = datasets.CIFAR10(root='./data', train=True, download=True, transform=transform_val)
val_indices = val_subset.indices
val_subset = torch.utils.data.Subset(val_dataset, val_indices)

test_dataset = datasets.CIFAR10(root='./data', train=False, download=True, transform=transform_val)

train_loader = DataLoader(train_subset, batch_size=128, shuffle=True, num_workers=2)
val_loader = DataLoader(val_subset, batch_size=256, shuffle=False, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=256, shuffle=False, num_workers=2)

print(f"Train: {len(train_subset)}, Val: {len(val_subset)}, Test: {len(test_dataset)}")


In [ ]:
# ============================================================
# Training Function
# ============================================================

def train_one_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss = 0
    correct = 0
    total = 0
    for X, y in loader:
        X, y = X.to(DEVICE), y.to(DEVICE)
        optimizer.zero_grad()
        out = model(X)
        loss = criterion(out, y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * X.size(0)
        correct += (out.argmax(1) == y).sum().item()
        total += X.size(0)
    return total_loss / total, correct / total

def evaluate(model, loader):
    model.eval()
    total_loss = 0
    correct = 0
    total = 0
    with torch.no_grad():
        for X, y in loader:
            X, y = X.to(DEVICE), y.to(DEVICE)
            out = model(X)
            loss = F.cross_entropy(out, y)
            total_loss += loss.item() * X.size(0)
            correct += (out.argmax(1) == y).sum().item()
            total += X.size(0)
    return total_loss / total, correct / total

def train_and_evaluate(model, train_loader, val_loader, test_loader, epochs=10, lr=0.01):
    """
    Train on train_loader, evaluate on val_loader for architecture ranking.
    Test set is reserved for final evaluation only.
    """
    optimizer = torch.optim.SGD(model.parameters(), lr=lr, momentum=0.9, weight_decay=5e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    criterion = nn.CrossEntropyLoss()
    
    history = {
        'train_loss': [], 'val_loss': [], 'val_acc': [],
        'test_loss': [], 'test_acc': [],
        'time': []
    }
    
    for epoch in range(epochs):
        t0 = time.time()
        train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, criterion)
        val_loss, val_acc = evaluate(model, val_loader)
        scheduler.step()
        elapsed = time.time() - t0
        
        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)
        history['time'].append(elapsed)
        
        print(f"  Epoch {epoch+1:2d}/{epochs}: "
              f"train={train_loss:.4f}, val={val_loss:.4f}/acc={val_acc:.4f} ({elapsed:.1f}s)")
    
    # Final test evaluation (only once at the end)
    test_loss, test_acc = evaluate(model, test_loader)
    history['test_loss'] = test_loss
    history['test_acc'] = test_acc
    
    return history


In [ ]:
# ============================================================
# Compute J_topo and Train Each Candidate
# ============================================================

results = []
t_start = time.time()

for i, config in enumerate(CANDIDATES):
    print(f"\n{'='*60}")
    print(f"[{i+1}/{len(CANDIDATES)}] {config['id']}: "
          f"depth={config['depth']}, wm={config['width_mult']}, skip={config['use_skip']}")
    
    # Build model
    torch.manual_seed(SEED)
    model = build_TN_arbitrary_depth(
        config['depth'], config['width_mult'], 10, config['use_skip']
    ).to(DEVICE)
    
    n_params = sum(p.numel() for p in model.parameters())
    print(f"  Parameters: {n_params/1e6:.2f}M")
    
    # Compute J_topo from fresh initialization
    J_topo = compute_J_topo(model)
    print(f"  J_topo (init): {J_topo:.4f}")
    
    # Train (uses val_loader for ranking)
    print(f"  Training {EPOCHS} epochs...")
    history = train_and_evaluate(model, train_loader, val_loader, test_loader, epochs=EPOCHS)
    
    # Record results
    results.append({
        'id': config['id'],
        'depth': config['depth'],
        'width_mult': config['width_mult'],
        'use_skip': config['use_skip'],
        'n_params': n_params,
        'J_topo': J_topo,
        'final_val_loss': history['val_loss'][-1],
        'final_val_acc': history['val_acc'][-1],
        'best_val_loss': min(history['val_loss']),
        'final_test_loss': history['test_loss'],
        'final_test_acc': history['test_acc'],
        'history': history,
    })
    
    print(f"  → Final: val_loss={history['val_loss'][-1]:.4f}, val_acc={history['val_acc'][-1]:.4f}")
    print(f"  → Test (reserved): loss={history['test_loss']:.4f}, acc={history['test_acc']:.4f}")

t_total = time.time() - t_start
print(f"\n{'='*60}")
print(f"Total time: {t_total/60:.1f} minutes")


In [ ]:
# ============================================================
# Results Summary
# ============================================================

print("\n" + "="*70)
print("RESULTS SUMMARY")
print("="*70)
print(f"{'ID':<6} {'J_topo':<10} {'Val Loss':<12} {'Val Acc':<12} {'Params':<10}")
print("-"*70)

# Sort by J_topo (prediction)
by_J = sorted(results, key=lambda x: -x['J_topo'])
for rank, r in enumerate(by_J, 1):
    print(f"{r['id']:<6} {r['J_topo']:<10.4f} {r['final_val_loss']:<12.4f} "
          f"{r['final_val_acc']:<12.4f} {r['n_params']/1e6:<10.2f}M")

print()

# Sort by validation loss (lower is better)
by_loss = sorted(results, key=lambda x: x['final_val_loss'])
print("Ranked by Validation Loss (lower = better):")
for rank, r in enumerate(by_loss, 1):
    print(f"  {rank}. {r['id']}: val_loss={r['final_val_loss']:.4f}, test_loss={r['final_test_loss']:.4f}")

print()

# Compute Spearman correlation between J_topo and val loss
from scipy.stats import spearmanr

J_vals = [r['J_topo'] for r in results]
val_loss_vals = [r['final_val_loss'] for r in results]

r_spearman, p_val = spearmanr(J_vals, val_loss_vals)
print(f"Spearman correlation (J_topo vs Val Loss): r = {r_spearman:.4f}, p = {p_val:.4f}")
print()

# Interpretation
print("INTERPRETATION:")
if r_spearman < 0:
    print(f"  r < 0: Higher J_topo → Lower val loss (consistent with theory!)")
else:
    print(f"  r > 0: Higher J_topo → Higher val loss (INCONSISTENT with theory)")
    
if abs(r_spearman) >= 0.7:
    print(f"  |r| >= 0.7: STRONG — J_topo ranking is RELIABLE")
elif abs(r_spearman) >= 0.5:
    print(f"  |r| >= 0.5: MODERATE — J_topo useful but not definitive")
else:
    print(f"  |r| < 0.5: WEAK — J_topo alone is INSUFFICIENT")

print()
print("Note: val_loss is used for architecture ranking. test_loss is reserved for final evaluation only.")


In [ ]:
# ============================================================
# Visualization
# ============================================================

import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Training curves (val loss)
ax1 = axes[0]
for r in results:
    ax1.plot(r['history']['val_loss'], label=f"{r['id']} (J={r['J_topo']:.3f})")
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Validation Loss')
ax1.set_title('Validation Loss vs Epoch')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Plot 2: J_topo vs Final Val Loss
ax2 = axes[1]
for r in results:
    ax2.scatter(r['J_topo'], r['final_val_loss'], s=100, label=r['id'])
    ax2.annotate(r['id'], (r['J_topo'], r['final_val_loss']), 
                 xytext=(5, 5), textcoords='offset points')
ax2.set_xlabel('J_topo (higher = better)')
ax2.set_ylabel('Validation Loss (lower = better)')
ax2.set_title(f'J_topo vs Val Loss (Spearman r={r_spearman:.3f})')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('/kaggle/working/phase_b_validation.png', dpi=150)
plt.show()

print("Figure saved to /kaggle/working/phase_b_validation.png")


In [ ]:
# ============================================================
# Save Results
# ============================================================

output = {
    'timestamp': datetime.now().isoformat(),
    'epochs': EPOCHS,
    'seed': SEED,
    'candidates': [
        {k: v for k, v in r.items() if k != 'history'} 
        for r in results
    ],
    'spearman': {'r': r_spearman, 'p': p_val},
    'total_time_minutes': t_total / 60,
}

out_path = '/kaggle/working/phase_b_quick_validation_results.json'
with open(out_path, 'w') as f:
    json.dump(output, f, indent=2)

print(f"Results saved to {out_path}")

In [ ]:
# ============================================================
# Visualization
# ============================================================

import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Training curves
ax1 = axes[0]
for r in results:
    ax1.plot(r['history']['test_loss'], label=f"{r['id']} (J={r['J_topo']:.3f})")
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Test Loss')
ax1.set_title('Test Loss vs Epoch')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Plot 2: J_topo vs Final Test Loss
ax2 = axes[1]
for r in results:
    ax2.scatter(r['J_topo'], r['final_test_loss'], s=100, label=r['id'])
    ax2.annotate(r['id'], (r['J_topo'], r['final_test_loss']), 
                 xytext=(5, 5), textcoords='offset points')
ax2.set_xlabel('J_topo (higher = better)')
ax2.set_ylabel('Test Loss (lower = better)')
ax2.set_title(f'J_topo vs Test Loss (Spearman r={r_spearman:.3f})')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('/kaggle/working/phase_b_validation.png', dpi=150)
plt.show()

print("Figure saved to /kaggle/working/phase_b_validation.png")